In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

import pickle

In [4]:
df = pd.read_csv('crime_dataset_india.csv') 
df.head()

,Report Number,Date Reported,Date of Occurrence,Time of Occurrence,City,Crime Code,Crime Description,Victim Age,Victim Gender,Weapon Used,Crime Domain,Police Deployed,Case Closed,Date Case Closed
0,1,02-01-2020 00:00,01-01-2020 00:00,01-01-2020 01:11,Ahmedabad,576,IDENTITY THEFT,16,M,Blunt Object,Violent Crime,13,No,NaN
1,2,01-01-2020 19:00,01-01-2020 01:00,01-01-2020 06:26,Chennai,128,HOMICIDE,37,M,Poison,Other Crime,9,No,NaN
2,3,02-01-2020 05:00,01-01-2020 02:00,01-01-2020 14:30,Ludhiana,271,KIDNAPPING,48,F,Blunt Object,Other Crime,15,No,NaN
3,4,01-01-2020 05:00,01-01-2020 03:00,01-01-2020 14:46,Pune,170,BURGLARY,49,F,Firearm,Other Crime,1,Yes,29-04-2020 05:00
4,5,01-01-2020 21:00,01-01-2020 04:00,01-01-2020 16:51,Pune,421,VANDALISM,30,F,Other,Other Crime,18,Yes,08-01-2020 21:00


In [7]:
df['Date Reported'] = pd.to_datetime(df['Date Reported'], format='mixed', dayfirst=True, errors='coerce')
df['Date of Occurrence'] = pd.to_datetime(df['Date of Occurrence'], format='mixed', dayfirst=True, errors='coerce')
df['Time of Occurrence'] = pd.to_datetime(df['Time of Occurrence'], format='mixed', dayfirst=True, errors='coerce')

In [8]:
df['report_delay'] = (df['Date Reported'] - df['Date of Occurrence']).dt.days
df['hour'] = df['Time of Occurrence'].dt.hour
df['day_of_week'] = df['Date of Occurrence'].dt.dayofweek

In [9]:
df.fillna(0, inplace=True)

In [10]:
categorical_cols = ['City','Crime Description','Weapon Used','Crime Domain','Victim Gender']

df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

In [11]:
df['Case Closed'] = df['Case Closed'].map({'Yes':1,'No':0})

X = df.drop(['Case Closed','Date Case Closed'], axis=1)
y = df['Case Closed']

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [16]:

df = df.drop([
    'Date Reported',
    'Date of Occurrence',
    'Time of Occurrence'
], axis=1)

df['Case Closed'] = df['Case Closed'].map({'Yes':1,'No':0})

X = df.drop(['Case Closed','Date Case Closed'], axis=1)
y = df['Case Closed']

In [18]:
df['Case Closed'].value_counts(dropna=False)

Case Closed
NaN    40160
Name: count, dtype: int64

In [19]:
df = df.dropna(subset=['Case Closed'])

In [20]:
X = df.drop(['Case Closed','Date Case Closed'], axis=1)
y = df['Case Closed']

In [22]:
print(df.shape)

(0, 68)


In [24]:
df = pd.read_csv('crime_dataset_india.csv')

In [25]:
df['Date Reported'] = pd.to_datetime(df['Date Reported'], format='mixed', dayfirst=True, errors='coerce')
df['Date of Occurrence'] = pd.to_datetime(df['Date of Occurrence'], format='mixed', dayfirst=True, errors='coerce')
df['Time of Occurrence'] = pd.to_datetime(df['Time of Occurrence'], format='mixed', dayfirst=True, errors='coerce')

In [26]:
df['Case Closed'] = df['Case Closed'].map({'Yes':1,'No':0})

df = df.dropna(subset=['Case Closed'])

In [27]:
df.fillna(0, inplace=True)

In [28]:
df['report_delay'] = (df['Date Reported'] - df['Date of Occurrence']).dt.days
df['hour'] = df['Time of Occurrence'].dt.hour
df['day_of_week'] = df['Date of Occurrence'].dt.dayofweek

In [29]:
df = df.drop([
    'Date Reported',
    'Date of Occurrence',
    'Time of Occurrence',
    'Date Case Closed'
], axis=1)

In [30]:
df = pd.get_dummies(df, drop_first=True)

In [31]:
X = df.drop('Case Closed', axis=1)
y = df['Case Closed']

In [32]:
print(X.shape)
print(y.shape)

(40160, 66)
(40160,)


In [33]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier(n_estimators=100)
model.fit(X_train, y_train)

RandomForestClassifier()

In [34]:
pred = model.predict(X_test)

In [35]:
from sklearn.metrics import accuracy_score, classification_report

print("Accuracy:", accuracy_score(y_test, pred))
print(classification_report(y_test, pred))

Accuracy: 0.48804780876494025
              precision    recall  f1-score   support

           0       0.49      0.53      0.51      4014
           1       0.49      0.45      0.47      4018

    accuracy                           0.49      8032
   macro avg       0.49      0.49      0.49      8032
weighted avg       0.49      0.49      0.49      8032



In [36]:
from sklearn.metrics import confusion_matrix

print(confusion_matrix(y_test, pred))

[[2108 1906]
 [2206 1812]]


In [37]:
import pandas as pd

importance = pd.Series(model.feature_importances_, index=X.columns)
print(importance.sort_values(ascending=False).head(10))

Report Number          0.117230
Crime Code             0.114279
Victim Age             0.099954
hour                   0.083800
Police Deployed        0.079173
report_delay           0.077332
day_of_week            0.056488
Victim Gender_M        0.016920
Weapon Used_Firearm    0.012427
Weapon Used_Knife      0.012372
dtype: float64


In [38]:
import pickle

pickle.dump(model, open('model.pkl','wb'))

In [39]:
pickle.dump(X.columns.tolist(), open('columns.pkl','wb'))